# 10 · Network & ISP Intelligence with IP2Location.io

**Final notebook in the 10-part IP2Location.io use-case series.** Enrich an example network-traffic CSV with ASN, autonomous system (AS), ISP, domain, and network usage type; compare traffic across networks; and export analyst-ready reports.

**Two modes:** `MODE = "demo"` is fully offline and uses **simulated network intelligence for teaching**. `MODE = "live"` queries the IP2Location.io REST API. The notebook does not pretend missing subscription fields are negative results or substitute an AS owner for a missing ISP.

**Input:** `sample_network_traffic.csv` (put it beside this `.ipynb`). **Outputs:** CSV reports in `outputs_notebook_10/`.

## 1. Install the packages

Run this in a separate notebook cell **only if needed** (or install the packages in your environment):

```python
%pip install pandas requests matplotlib
```

Python 3.9+ is recommended. You can execute this notebook from top to bottom using **Kernel → Restart & Run All**.

In [ ]:
import os
import time
import ipaddress
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

print("Libraries loaded.")

## 2. API capabilities and plan awareness

The **Free** response includes `asn` and `as` (autonomous system name); **Starter and above** provide fields such as `isp`, `domain`, `net_speed`, and `usage_type`; **Security** adds detailed `as_info` (`as_number`, `as_name`, `as_domain`, `as_usage_type`, `as_cidr`). Some fields may be absent even on eligible plans.

Use `as` / `as_info.as_name` for **AS ownership**, not as a stand-in for `isp`. `usage_type` refers to IP-range usage, while `as_info.as_usage_type` describes the AS; the two can differ.

Documentation: https://www.ip2location.io/ip2location-documentation

The single-IP endpoint used here is `https://api.ip2location.io/`. It is **not** the separate Bulk IP Geolocation API.

In [ ]:
# Shared configuration: ALL definitions required in live and demo mode are here.
MODE = "demo"                 # "demo" or "live"
INPUT_CSV = Path("sample_network_traffic.csv")
OUTPUT_DIR = Path("outputs_notebook_10")
API_ENDPOINT = "https://api.ip2location.io/"
API_KEY = os.getenv("IP2LOCATION_API_KEY", "").strip()
MAX_UNIQUE_LOOKUPS = 25       # safety cap; raise intentionally for larger datasets
REQUEST_DELAY_SECONDS = 0.15  # small courtesy delay between unique IP requests
HTTP_TIMEOUT_SECONDS = 15
SERVER_RETRIES = 1           # retries only on 5xx/transient request failures

INPUT_COLUMNS = ["event_id", "timestamp_utc", "source_ip", "endpoint", "channel", "request_count", "bytes_sent"]
LOOKUP_COLUMNS = [
    "lookup_status", "lookup_error", "data_source", "ip_version", "country_code", "country_name",
    "asn", "as_name", "as_domain", "as_cidr", "as_usage_type", "isp", "isp_domain",
    "usage_type", "net_speed", "address_type", "network_segment", "segment_basis", "detail_level"
]
USAGE_LABELS = {
    "DCH": "Hosting / Data Center", "CDN": "Content Delivery Network",
    "ISP": "Fixed-Line ISP", "MOB": "Mobile ISP", "ISP/MOB": "Mixed ISP",
    "COM": "Organization / Institutional", "ORG": "Organization / Institutional",
    "GOV": "Organization / Institutional", "MIL": "Organization / Institutional",
    "EDU": "Organization / Institutional", "LIB": "Organization / Institutional",
    "SES": "Automated Service", "AIC": "Automated Service", "RSV": "Reserved / Other"
}
assert MODE in {"demo", "live"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Mode:", MODE, "| Input:", INPUT_CSV)

## 3. Read and validate the traffic data

Only globally routable public addresses are queried. Private, loopback, link-local, reserved/documentation, and invalid addresses stay in the output with a descriptive status. Repeat IPs are queried once per run and joined back to **all original events**.

In [ ]:
def classify_ip(raw):
    try:
        addr = ipaddress.ip_address(str(raw).strip())
    except ValueError:
        return None, "invalid_ip", None
    if not addr.is_global:
        return str(addr), "non_public_ip", addr.version
    return str(addr), "public", addr.version

traffic = pd.read_csv(INPUT_CSV, dtype={"source_ip": str, "event_id": str}, keep_default_na=False)
missing = set(INPUT_COLUMNS) - set(traffic.columns)
if missing:
    raise ValueError(f"Input CSV missing columns: {sorted(missing)}")
traffic["request_count"] = pd.to_numeric(traffic["request_count"], errors="raise").astype(int)
traffic["bytes_sent"] = pd.to_numeric(traffic["bytes_sent"], errors="raise").astype(int)
if (traffic[["request_count", "bytes_sent"]] < 0).any().any():
    raise ValueError("request_count and bytes_sent must be nonnegative")
if traffic["event_id"].duplicated().any():
    raise ValueError("event_id must be unique for accurate output joins")
traffic["source_ip"] = traffic["source_ip"].astype(str).str.strip()
traffic[["normalized_ip", "input_ip_status", "ip_version"]] = pd.DataFrame(
    traffic["source_ip"].map(classify_ip).tolist(), index=traffic.index
)
public_ips = list(dict.fromkeys(traffic.loc[traffic.input_ip_status == "public", "normalized_ip"]))
print(f"Input events: {len(traffic)} | Public unique IPs: {len(public_ips)} | Non-public/invalid events: {(traffic.input_ip_status != 'public').sum()}")
display(traffic.head(8))

## 4. Offline demonstration fixtures (optional in live mode)

The following synthetic results are **not real assertions about the sample IPs**. We intentionally use example AS/ISP names to make the distinction between network data and real-world lookup results unmistakable. Feel free to skip this cell entirely in live mode: all shared definitions were initialized earlier.

In [ ]:
if MODE == "demo":
    demo_profiles = [
        {"country_code":"US", "country_name":"Demo Region West", "asn":"64510", "as":"Demo Cloud AS", "isp":"Example Cloud Hosting", "domain":"cloud.demo", "net_speed":"T1", "usage_type":"DCH",
         "as_info":{"as_number":"64510","as_name":"Demo Cloud AS","as_domain":"as-cloud.demo","as_usage_type":"DCH","as_cidr":"100.64.0.0/10"}},
        {"country_code":"MY", "country_name":"Demo Region East", "asn":"64511", "as":"Demo Telecom AS", "isp":"Example Fixed-Line Broadband", "domain":"broadband.demo", "net_speed":"DSL", "usage_type":"ISP", "as_info":{}},
        {"country_code":"SG", "country_name":"Demo Region South", "asn":"64512", "as":"Demo Mobile AS", "isp":"Example Mobile Carrier", "domain":"mobile.demo", "net_speed":"DSL", "usage_type":"MOB", "as_info":{}},
        {"country_code":"DE", "country_name":"Demo Region Central", "asn":"64513", "as":"Demo Delivery AS", "isp":"Example CDN Edge", "domain":"cdn.demo", "net_speed":"T1", "usage_type":"CDN", "as_info":{}},
        {"country_code":"GB", "country_name":"Demo Region North", "asn":"64514", "as":"Demo Research AS", "isp":"Example Campus Network", "domain":"campus.demo", "usage_type":"EDU", "as_info":{}},
        {"country_code":"JP", "country_name":"Demo Region Pacific", "asn":"64515", "as":"Demo Auto AS", "isp":"Example Search Crawler", "usage_type":"SES", "as_info":{}},
        {"country_code":"US", "country_name":"Demo Region West", "asn":"64516", "as":"Demo Mixed AS", "isp":"Example Combined ISP", "usage_type":"ISP/MOB", "as_info":{}},
        {"country_code":"CA", "country_name":"Demo Region North", "asn":"64517", "as":"Demo Unclassified AS"},
        {"country_code":"FR", "country_name":"Demo Region West", "asn":"64518", "as":"Demo AS with Detailed Metadata", "as_info":{"as_usage_type":"DCH", "as_domain":"sample-as.demo", "as_cidr":"198.18.0.0/15"}},
    ]
    DEMO_LOOKUPS = {ip: dict(demo_profiles[i % len(demo_profiles)], ip=ip)
                    for i, ip in enumerate(public_ips)}
    print("Offline teaching fixtures prepared:", len(DEMO_LOOKUPS))

## 5. Normalize API responses without guessing missing fields

Never equate a **missing** `usage_type`, `isp`, or `as_info` with a specific network type. We use an explicit **Unknown / Not Available** segment. If only AS-level usage is present, the notebook can show a segment inferred from `as_info.as_usage_type`, tagged `AS usage type (not IP usage)` to avoid implying IP-range certainty.

In [ ]:
def clean_field(value):
    if value is None or isinstance(value, (list, dict)):
        return None
    v = str(value).strip()
    return None if v in {"", "-", "N/A", "null", "None"} else v

def normalize_response(ip, data, source):
    if not isinstance(data, dict):
        return {"lookup_status": "api_error", "lookup_error": "Response is not a JSON object"}
    as_info = data.get("as_info") if isinstance(data.get("as_info"), dict) else {}
    if isinstance(data.get("error"), dict):
        error = data["error"]
        code = str(error.get("error_code", ""))
        status = "quota_or_auth_error" if code == "10000" else "api_error"
        return {"lookup_status": status, "lookup_error": f"API error {code}: {error.get('error_message', '')}"}
    asn = clean_field(data.get("asn")) or clean_field(as_info.get("as_number"))
    as_name = clean_field(data.get("as")) or clean_field(as_info.get("as_name"))
    isp = clean_field(data.get("isp"))
    usage = clean_field(data.get("usage_type"))
    as_usage = clean_field(as_info.get("as_usage_type"))
    if usage:
        code = usage.upper()
        network_segment = USAGE_LABELS.get(code, f"Other: {code}")
        basis = "IP usage type"
    elif as_usage:
        code = as_usage.upper()
        network_segment = USAGE_LABELS.get(code, f"Other: {code}")
        basis = "AS usage type (not IP usage)"
    else:
        network_segment = "Unknown / Not Available"
        basis = "Not available"
    detail_level = "AS detail available" if as_info else ("ISP fields available" if isp or usage else "ASN / AS only")
    return {
        "lookup_status": "ok", "lookup_error": "", "data_source": source,
        "country_code": clean_field(data.get("country_code")),
        "country_name": clean_field(data.get("country_name")),
        "asn": asn, "as_name": as_name,
        "as_domain": clean_field(as_info.get("as_domain")),
        "as_cidr": clean_field(as_info.get("as_cidr")),
        "as_usage_type": as_usage, "isp": isp, "isp_domain": clean_field(data.get("domain")),
        "usage_type": usage, "net_speed": clean_field(data.get("net_speed")),
        "address_type": clean_field(data.get("address_type")),
        "network_segment": network_segment, "segment_basis": basis,
        "detail_level": detail_level,
    }

# Boundary tests protect the Free-plan and Security-plan response shapes.
assert normalize_response("8.8.8.8", {"asn":"15169", "as":"Example AS"}, "test")["network_segment"] == "Unknown / Not Available"
assert normalize_response("8.8.8.8", {"usage_type":"DCH"}, "test")["network_segment"] == "Hosting / Data Center"
assert normalize_response("8.8.8.8", {"as_info":{"as_usage_type":"CDN"}}, "test")["segment_basis"] == "AS usage type (not IP usage)"
print("Response parser tests passed.")

## 6. Live REST lookups with graceful API failure handling

This is the **single-IP geolocation** API. Authentication uses a Bearer token if `IP2LOCATION_API_KEY` is set; otherwise it makes a keyless request. Requests are bounded by `MAX_UNIQUE_LOOKUPS`, use a timeout, retry only transient server failures, and halt after a quota/authentication failure. No API calls occur in demo mode.

In [ ]:
def fetch_live_ip(ip, session):
    headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
    for attempt in range(SERVER_RETRIES + 1):
        try:
            resp = session.get(API_ENDPOINT, params={"ip": ip, "format": "json"}, headers=headers, timeout=HTTP_TIMEOUT_SECONDS)
            if resp.status_code in {401, 403, 429}:
                return {"lookup_status": "quota_or_auth_error", "lookup_error": f"HTTP {resp.status_code}", "data_source": "live"}
            if 500 <= resp.status_code < 600:
                if attempt < SERVER_RETRIES:
                    time.sleep(0.25 * (attempt + 1))
                    continue
                return {"lookup_status": "api_error", "lookup_error": f"HTTP {resp.status_code}", "data_source": "live"}
            if resp.status_code >= 400:
                return {"lookup_status": "api_error", "lookup_error": f"HTTP {resp.status_code}", "data_source": "live"}
            payload = resp.json()
            return normalize_response(ip, payload, "live")
        except (requests.exceptions.RequestException, ValueError) as exc:
            if attempt < SERVER_RETRIES:
                time.sleep(0.25 * (attempt + 1))
                continue
            return {"lookup_status": "request_error", "lookup_error": type(exc).__name__, "data_source": "live"}


def enrich_unique_public_ips(ip_list, session=None):
    lookups = []
    stop_reason = None
    if MODE == "live" and session is None:
        session = requests.Session()
    for idx, ip in enumerate(ip_list):
        if stop_reason:
            item = {"lookup_status": "skipped_after_quota", "lookup_error": stop_reason, "data_source": MODE}
        elif idx >= MAX_UNIQUE_LOOKUPS:
            item = {"lookup_status": "skipped_limit", "lookup_error": "Exceeded MAX_UNIQUE_LOOKUPS", "data_source": MODE}
        elif MODE == "demo":
            item = normalize_response(ip, DEMO_LOOKUPS[ip], "demo (synthetic)")
        else:
            item = fetch_live_ip(ip, session)
            if item.get("lookup_status") == "quota_or_auth_error":
                stop_reason = item.get("lookup_error") or "Quota/authentication error"
            if idx + 1 < min(len(ip_list), MAX_UNIQUE_LOOKUPS) and not stop_reason:
                time.sleep(REQUEST_DELAY_SECONDS)
        lookups.append({"normalized_ip": ip, **item})
    return pd.DataFrame(lookups, columns=["normalized_ip"] + LOOKUP_COLUMNS)

print("Live request and deduplication helpers ready.")

## 7. Run lookup and join back to traffic events

Every input event remains represented in the enriched output. For invalid or non-public IPs we preserve the reason and leave network metadata null, rather than inventing location or ISP values.

In [ ]:
lookup_df = enrich_unique_public_ips(public_ips)
output = traffic.merge(lookup_df, on="normalized_ip", how="left", validate="many_to_one", suffixes=("", "_api"))
output["lookup_status"] = output["lookup_status"].fillna(output["input_ip_status"])
output["lookup_error"] = output["lookup_error"].fillna("")
output.loc[output.input_ip_status != "public", "lookup_status"] = output.loc[output.input_ip_status != "public", "input_ip_status"]
output.loc[output.input_ip_status != "public", "lookup_error"] = "Not sent to API"
output["network_segment"] = output["network_segment"].fillna("Unknown / Not Available")
output["segment_basis"] = output["segment_basis"].fillna("Not available")
output["data_source"] = output["data_source"].fillna("not queried")
assert len(output) == len(traffic)
assert output["event_id"].nunique() == len(traffic)
assert output["request_count"].sum() == traffic["request_count"].sum()
print("Rows:", len(output), "| Successful event enrichments:", (output.lookup_status == "ok").sum())
display(output[["event_id", "source_ip", "lookup_status", "asn", "as_name", "isp", "usage_type", "network_segment", "segment_basis"]].head(12))

## 8. Network traffic summaries

**Requests are weighted** using the `request_count` field, not just the number of CSV rows. Separately report unique IPs, which should not be confused with unique users, and bytes transferred, which are sample values in this tutorial.

Use `Unknown / Not Available` as its own category: a Free-tier lack of `usage_type` should not turn into a false "residential" or "safe" inference.

In [ ]:
def summarize(group_columns, source=output):
    return (source.groupby(group_columns, dropna=False)
            .agg(events=("event_id", "size"), requests=("request_count", "sum"),
                 unique_ips=("normalized_ip", "nunique"), bytes_sent=("bytes_sent", "sum"))
            .reset_index().sort_values(["requests", "events"], ascending=[False, False]))

segment_summary = summarize(["network_segment", "segment_basis"])
segment_summary["share_of_requests_pct"] = (100 * segment_summary["requests"] / max(output.request_count.sum(), 1)).round(2)
asn_summary = summarize(["asn", "as_name"])
isp_summary = summarize(["isp", "isp_domain"])
country_summary = summarize(["country_code", "country_name"])
endpoint_summary = summarize(["endpoint", "network_segment"])
print("Network usage:")
display(segment_summary)
print("Top AS operators (not necessarily the ISP):")
display(asn_summary.head(10))
print("ISP coverage and top providers:")
display(isp_summary.head(10))

## 9. Visual exploration

The charts summarize the traffic dataset, not internet-wide ownership. Unknown categories remain visible. The synthetic demo is for illustration, **not an assertion about any actual IP**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_data = segment_summary.groupby("network_segment", dropna=False)["requests"].sum().sort_values(ascending=True)
axes[0].barh(plot_data.index.astype(str), plot_data.values)
axes[0].set_title("Requests by network segment")
axes[0].set_xlabel("Requests")

asn_plot = asn_summary.copy().head(8)
asn_plot["label"] = asn_plot["asn"].fillna("Unknown") + " · " + asn_plot["as_name"].fillna("Not available")
axes[1].barh(asn_plot["label"].iloc[::-1], asn_plot["requests"].iloc[::-1])
axes[1].set_title("Top ASNs by requests")
axes[1].set_xlabel("Requests")
plt.tight_layout()
plt.show()

## 10. Completeness and lookup status audit

The report below reveals how many successfully queried IPs have optional fields. **Missing ISP/usage data may reflect your subscription, unavailable source data, or both.** An `asn` or `as` alone is not enough to identify a consumer connection or make a security decision.

In [ ]:
status_summary = (output.groupby("lookup_status", dropna=False)
                  .agg(events=("event_id", "size"), requests=("request_count", "sum"), unique_ips=("source_ip", "nunique"))
                  .reset_index().sort_values("events", ascending=False))

ok_unique = lookup_df[lookup_df.lookup_status == "ok"].drop_duplicates("normalized_ip") if not lookup_df.empty else pd.DataFrame()
coverage_records = []
for field in ["asn", "as_name", "isp", "isp_domain", "usage_type", "as_domain", "as_cidr", "as_usage_type"]:
    populated = int(ok_unique[field].notna().sum()) if (not ok_unique.empty and field in ok_unique) else 0
    total = len(ok_unique)
    coverage_records.append({"field": field, "unique_public_ips_looked_up": total, "populated": populated,
                             "coverage_pct": round(100 * populated / total, 2) if total else 0.0})
coverage_summary = pd.DataFrame(coverage_records)
display(status_summary)
display(coverage_summary)

## 11. Export the CSV reports

The CSV outputs are intended for inspection and downstream analysis. `network_traffic_enriched.csv` contains all original rows, including invalid and private addresses; reports use request-weighted aggregations. Existing files in the output folder are overwritten when the notebook runs.

In [ ]:
reports = {
    "network_traffic_enriched.csv": output,
    "network_segment_summary.csv": segment_summary,
    "network_asn_summary.csv": asn_summary,
    "network_isp_summary.csv": isp_summary,
    "network_country_summary.csv": country_summary,
    "network_endpoint_segment_summary.csv": endpoint_summary,
    "network_lookup_status_summary.csv": status_summary,
    "network_field_coverage.csv": coverage_summary,
}
for filename, frame in reports.items():
    target = OUTPUT_DIR / filename
    frame.to_csv(target, index=False)
    print(f"Saved {target} ({len(frame)} rows)")

## 12. Interpretation and limitations

- **ASN / AS ≠ ISP:** the AS operator and the provider for a particular IP range may differ. Interpret them separately.
- **Usage types are classifications**, not proof that a person is residential/mobile, or that their traffic is malicious. `ISP/MOB` is an explicitly mixed category.
- **Plan-aware:** Free supports ASN and AS names. Starter adds ISP and usage types; Security adds detailed AS information. Do not substitute unknown fields with synthetic guesses in live mode.
- **Demo is synthetic:** its records and fixtures must not be published as genuine IP2Location.io lookup results.
- **Privacy:** process traffic logs only where you have lawful permission and follow your retention and access policies.
- **Quotas and availability:** set a safe lookup cap, deduplicate, and handle API errors. For higher volume, evaluate the separate bulk product and its plan requirements.

**Source:** https://www.ip2location.io/ip2location-documentation

**End of the 10-notebook series.**